# Example: BBBC021 — Mechanism of Action (MoA) Profiling

Perform morphological profiling on a multi-channel compound screen to analyze and predict the Mechanism of Action (MoA) of various treatments.

**Dataset:** [BBBC021](https://bbbc.broadinstitute.org/BBBC021) (3-channel fluorescent compound screen)

**Key Steps:**
1. **Metadata Integration:** Merge image metadata with compound MoA annotations.
2. **Multi-Channel Processing:** Link and process 3-channel images (Actin, Tubulin, DAPI).
3. **MoA Analysis:** Extract deep embeddings, compute extended properties, and classify compound effects within the phenospace.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Define the paths to the BBBC021 dataset and the metadata files, including the Mechanism of Action (MoA) annotations.

In [ ]:
IMAGE_DIR = "/datasets/BBBC/BBBC021/Week123"
METADATA_PATH = "/datasets/BBBC/BBBC021/metadata.csv"

## 2. Load & Merge Metadata

We load the image metadata and merge it with the MoA table to associate each treatment with its known biological mechanism.

In [4]:
df_merged = pd.read_csv(METADATA_PATH)
df_merged.head()

,TableNumber,ImageNumber,Image_FileName_DAPI,Image_PathName_DAPI,Image_FileName_Tubulin,Image_PathName_Tubulin,Image_FileName_Actin,Image_PathName_Actin,Image_Metadata_Plate_DAPI,Image_Metadata_Well_DAPI,Replicate,Image_Metadata_Compound,Image_Metadata_Concentration,Mechanism_of_action
0,1,11,Week1_150607_B04_s3_w135D66B4C-0548-4AB8-A57B-...,Week1/Week1_22123,Week1_150607_B04_s3_w2D81AEFB9-9DD4-4B59-9177-...,Week1/Week1_22123,Week1_150607_B04_s3_w408BE006A-BF34-457E-81A9-...,Week1/Week1_22123,Week1_22123,B04,1,cytochalasin B,10.0,Actin disruptors
1,1,12,Week1_150607_B04_s4_w1EB720AD1-83BA-48A7-8C47-...,Week1/Week1_22123,Week1_150607_B04_s4_w261B79A05-7534-46F3-8C80-...,Week1/Week1_22123,Week1_150607_B04_s4_w49FF7E7B1-F049-4994-BCA2-...,Week1/Week1_22123,Week1_22123,B04,1,cytochalasin B,10.0,Actin disruptors
2,1,9,Week1_150607_B04_s1_w11323931B-BDA7-4F42-870E-...,Week1/Week1_22123,Week1_150607_B04_s1_w2F8F7EA7A-EC57-49CA-A556-...,Week1/Week1_22123,Week1_150607_B04_s1_w494DCA5C4-3531-497D-A8B0-...,Week1/Week1_22123,Week1_22123,B04,1,cytochalasin B,10.0,Actin disruptors
3,1,10,Week1_150607_B04_s2_w1F649C703-6FA8-406F-8575-...,Week1/Week1_22123,Week1_150607_B04_s2_w2802CC81E-56F9-41C5-A6C2-...,Week1/Week1_22123,Week1_150607_B04_s2_w4342F300D-60F8-4256-A637-...,Week1/Week1_22123,Week1_22123,B04,1,cytochalasin B,10.0,Actin disruptors
4,1,251,Week1_150607_B04_s3_w1B0B8AAC7-F2AD-4756-939C-...,Week1/Week1_22141,Week1_150607_B04_s3_w25C7FD77E-FDC4-471F-A512-...,Week1/Week1_22141,Week1_150607_B04_s3_w42BD03411-D32F-4179-A6C5-...,Week1/Week1_22141,Week1_22141,B04,2,cytochalasin B,10.0,Actin disruptors


## 3. Multi-Channel Data Discovery

BBBC021 consists of 3-channel images (Actin, Tubulin, DAPI). We specify the filename columns for each channel to ensure they are correctly linked and loaded. `channel_names` uses the same order so property columns are labelled by stain (e.g. `intensity_mean_Actin`) instead of `_ch0`.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    IMAGE_DIR,
    metadata_fn=make_dataframe_metadata_fn(
        df_merged,
        filename_column=["Image_FileName_Actin", "Image_FileName_Tubulin", "Image_FileName_DAPI"],
    ),
    channel_names=["Actin", "Tubulin", "DAPI"],
)
file_df.head()

## 4. Feature Extraction

We extract DINOv2 embeddings and compute an extended set of morphological properties to capture subtle phenotypic changes induced by the compounds.

> **Tip — Checkpoints & Resuming**:
> - **Save / Resume during extraction**: Pass `checkpoint_path="embeddings.h5"` to `process_images` and `checkpoint_path="properties.h5"` to `compute_properties`. Embeddings and properties are written incrementally to HDF5, saving RAM and enabling automatic resume if interrupted.
> - **Load precomputed checkpoints**: If checkpoints were already computed, skip feature extraction entirely by loading them with `pheno.load_embeddings(...)` and `pheno.load_properties(...)` (or `pheno.load_results(...)`).

In [ ]:
# Option A: Load precomputed checkpoints (skip extraction)
# pheno.load_embeddings("/path/to/bbbc021/.../embeddings.h5")
# pheno.load_properties("/path/to/bbbc021/.../properties.h5")

# Option B: Extract features (optionally add checkpoint_path to save/resume)
pheno.process_images(
    wrapper,
    batch_size=16,
    channel_mode="split",
    resize_size=1036,
    pad_size=1280,
    l2_normalize_channels=False,
    # checkpoint_path="bbbc021_embeddings.h5",  # Save embeddings incrementally to HDF5
)

df_properties = pheno.compute_properties(
    property_preset="complete",
    n_jobs=4,
    # checkpoint_path="bbbc021_properties.h5",  # Save properties incrementally to HDF5
)

Found 1080 committed images; skipping 1080 (already in checkpoint), 0 to process.
All images already processed. Loading from checkpoint.
Loaded Processing pipeline:
  1. Read images: native dtype uint16, shape 1024x1280x3, range 0-6.341e+04
  2. Layout to height x width x channels: channel axis inferred (3 channels)
  3. Channel selection: all channels
  4. Channel handling: split; each channel processed separately, grayscale expanded to RGB
  5. Scale intensities: divide by 65535 to reach 0-1
  6. Pad: minimum size 1280x1280, black (constant fill)
  7. Resize: 1036x1036, bilinear (antialiased)
  8. Normalize for the model: ImageNet mean and standard deviation (3 channel(s))
  9. Vision model: DinoV2ModelWrapper (dinov2_vitg14_reg) → embedding
  10. Embedding adjustment: concatenate channel embeddings without L2 (magnitudes preserved)
Pipeline ready: 1080 images available (lazy loading from /data3/agonzalez/checkpoints/txukun/BBBC021_split_checkpoint_vitg_embeddings.h5).
Property proce

## 5. Group Statistics

Aggregate morphological properties by compound and Mechanism of Action to identify characteristic phenotypic profiles.

In [8]:
df_grouped = pheno.property_stats_by_group(
    group_by=["Image_Metadata_Compound", "Mechanism_of_action"],
    print_properties=["intensity_mean_Actin", "intensity_std_Actin"],
)


                    PROPERTY STATISTICS BY IMAGE_METADATA_COMPOUND AND MECHANISM_OF_ACTION                   
 Image_metadata_compound    Mechanism_of_action    N  |  Intensity_mean... (μ±σ)  |  Intensity_std_... (μ±σ) 
-------------------------------------------------------------------------------------------------------------
     cytochalasin B          Actin disruptors     24  |         0.02±0.00         |         0.03±0.00        
      latrunculin B          Actin disruptors     24  |         0.02±0.00         |         0.02±0.00        
          taxol           Microtubule stabilizers 468 |         0.03±0.01         |         0.04±0.02        
      epothilone B        Microtubule stabilizers 36  |         0.02±0.00         |         0.02±0.00        
       nocodazole        Microtubule destabilizers24  |         0.03±0.00         |         0.03±0.00        
       colchicine        Microtubule destabilizers12  |         0.03±0.01         |         0.02±0.00        
         

## 6. Visualization

Visualize the compound phenospace using t-SNE, colored by Mechanism of Action, to observe how different treatments cluster based on their morphological effects.

In [9]:
pheno.plot_tsne(
    color_by="mechanism_of_action",
    n_components=2,
    exclude={"Image_metadata_compound": "taxol"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)

Computing t-SNE (612 samples, 4608 features → 2 components)...


In [10]:
pheno.plot_tsne(
    color_by="image_metadata_compound",
    n_components=2,
    filters={"mechanism_of_action": "Microtubule destabilizers"},
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)


Computing t-SNE (168 samples, 4608 features → 2 components)...


## 7. Correlation Analysis
Compute correlations between morphological features and their embeddings to identify potential relationships.

In [11]:
corr_results = pheno.compute_component_correlation(
    method="tsne",
    n_components=2,
    source="embeddings",
    exclude={"Image_Metadata_Compound": "taxol"},
    filters={"Mechanism_of_action": "Microtubule destabilizers"},
    top_k=5,
    correlation_method="pearson",
)

Computing t-SNE (168 samples, 4608 features → 2 components)...
Computing Pearson correlations (18 properties x 2 components)...


## 8. Interactive Exploration

Launch the interactive explorer to visually inspect the 3-channel images and explore the relationship between compound concentration and phenotypic response.

In [ ]:
e = pheno.create_interactive_explorer(
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)